In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load dataset (MovieLens u.data)
df = pd.read_csv("u.data", sep="\t", header=None, names=["user_id", "item_id", "rating", "timestamp"])
df = df.drop(columns=["timestamp"])  # Timestamp not needed

# Convert to 0-based indices (Python-style)
df["user_id"] -= 1
df["item_id"] -= 1

# Get number of users and items
num_users = df["user_id"].max() + 1
num_items = df["item_id"].max() + 1

# Train-test split (90% train, 10% test)
train_data, test_data = train_test_split(df, test_size=0.25, random_state=42)
train_tuples = list(train_data.itertuples(index=False, name=None))
test_tuples = list(test_data.itertuples(index=False, name=None))

In [20]:
class DecentralizedMF:
    def __init__(self, num_users, num_items, latent_dim=10, lr=0.01, alpha=0.01, beta=0.01, gamma=0.01):
        self.num_users = num_users
        self.num_items = num_items
        self.latent_dim = latent_dim
        self.lr = lr
        self.alpha = alpha  # regularization for user latent factors
        self.beta = beta    # regularization for global item latent factors
        self.gamma = gamma  # regularization for personal item latent factors

        # User latent factors: u_i
        self.user_factors = np.random.normal(scale=0.1, size=(num_users, latent_dim))

        # Global item latent factors: p_j
        self.common_item_factors = np.random.normal(scale=0.1, size=(num_items, latent_dim))

        # Personal item latent factors: q_ij
        self.personal_item_factors = np.random.normal(scale=0.1, size=(num_users, num_items, latent_dim))

    def train(self, data, epochs=10):
        for epoch in range(epochs):
            np.random.shuffle(data)
            for user_id, item_id, rating in data:
                u = self.user_factors[user_id]
                p = self.common_item_factors[item_id]
                q = self.personal_item_factors[user_id, item_id]

                # Prediction
                pred = np.dot(u, p + q)
                error = rating - pred

                # Gradients
                grad_u = -error * (p + q) + self.alpha * u
                grad_p = -error * u + self.beta * p
                grad_q = -error * u + self.gamma * q

                # Update
                self.user_factors[user_id] -= self.lr * grad_u
                self.common_item_factors[item_id] -= self.lr * grad_p
                self.personal_item_factors[user_id, item_id] -= self.lr * grad_q

    def predict(self, user_id, item_id):
        u = self.user_factors[user_id]
        p = self.common_item_factors[item_id]
        q = self.personal_item_factors[user_id, item_id]
        return np.dot(u, p + q)

    def compute_rmse(self, data):
        mse = 0.0
        for user_id, item_id, rating in data:
            pred = self.predict(user_id, item_id)
            mse += (rating - pred) ** 2
        return np.sqrt(mse / len(data))

In [22]:
# Instantiate the model
dmf_model = DecentralizedMF(num_users=num_users, num_items=num_items, latent_dim=10, lr=0.01)

# Train the model
dmf_model.train(train_tuples, epochs=10)

# Evaluate on test set
rmse = dmf_model.compute_rmse(test_tuples)
print(f"Test RMSE: {rmse:.4f}")

Test RMSE: 1.0048
